# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library while referencing all dataset entities by their `@id` as required by the Croissant schema.

### Dataset Source
The dataset is described by the following Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed for data loading
!pip install mlcroissant --quiet

## 1. Data Loading
Load Croissant metadata and records using the `mlcroissant` API.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset as a Croissant object
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Loaded dataset: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review the available record sets, fields, and their Croissant `@id`s. We'll display all record set `@id`s and, for each, the field `@id`s that can be loaded.


In [ ]:
from pprint import pprint

def recordsets_overview(ds):
    print(f"Dataset record sets (@id):")
    for rs in ds.metadata.record_sets:
        print(f"- Record Set: {rs['@id']}")
        if 'fields' in rs:
            print("  Fields:")
            for f in rs['fields']:
                print(f"    - {f['@id']}")
        print('')

recordsets_overview(dataset)


## 3. Data Extraction
Extract data from each record set into pandas DataFrames for further analysis.

**Note:** In this notebook, all Croissant entities are referenced by their `@id`. Use the output above to select the `@id` for each record set and desired fields.


In [ ]:
# Gather all record set @ids from metadata
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

# Load the first 3 records from each record set as a sample
for rs_id in record_set_ids:
    records_iter = dataset.records(record_set=rs_id)
    records = []
    try:
        for i, r in enumerate(records_iter):
            records.append(r)
            if i >= 2:
                break
    except Exception as e:
        print(f"Error loading records from {rs_id}: {e}")
    dataframes[rs_id] = pd.DataFrame(records)

# Preview columns and head of the first available record set loaded
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Record Set @id: {first_rs}")
    print("Columns:", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Perform common data processing: filter by a numeric column, normalize, and group. All field references use their Croissant `@id`.


In [ ]:
# For example/demo purposes, pick the first record set if available
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes.get(record_set_id, pd.DataFrame())
    
    if not df.empty:
        # Attempt to locate a numeric field (float or int).
        # We assume the field IDs correspond to DataFrame column names.
        numeric_field = None
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break

        if numeric_field is not None:
            threshold = df[numeric_field].quantile(0.75)
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records from record set '{record_set_id}' where '{numeric_field}' > {threshold}:")
            display(filtered_df.head())

            # Normalize
            normalized_col = f"{numeric_field}_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
            print(f"Normalized '{numeric_field}' (column '{normalized_col}'):")
            display(filtered_df[[numeric_field, normalized_col]].head())

            # Try grouping by another column (pick a non-numeric one, if available)
            group_field = None
            for col in df.columns:
                if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                    group_field = col
                    break

            if group_field is not None:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(f"Grouped filtered data by field '{group_field}':")
                display(grouped_df.head())
            else:
                print("No categorical field found for grouping.")
        else:
            print(f"No numeric field found in record set '{record_set_id}'.")
    else:
        print(f"DataFrame for record set '{record_set_id}' is empty.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field, grouped by a categorical field, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and not df.empty and (numeric_field is not None):
    plt.figure(figsize=(7,4))
    if group_field is not None:
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Distribution of '{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
    else:
        sns.histplot(df[numeric_field], kde=True)
        plt.title(f"Distribution of '{numeric_field}'")
        plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
else:
    print("Unable to plot due to missing data or fields.")

## 6. Conclusion
This notebook demonstrated:
- Loading and exploring a Croissant-structured dataset with `mlcroissant`
- Referencing all dataset entities by their `@id`
- Extracting and examining data with pandas
- Performing basic EDA and visualization

Further analyses can be performed on this dataset by building on these steps, consistently referencing the `@id` for record sets and fields for reproducibility and compatibility with Croissant tooling.
